# Lab: The Bernstein-Vazirani Algorithm with Qiskit

In this lab we implement the **Bernstein-Vazirani algorithm**, one of the first examples of a quantum algorithm that beats any classical algorithm for the same task.

We are given a black-box ("oracle") function

$$f: \{0, 1\}^n \rightarrow \{0, 1\}, \qquad f(x) = s \cdot x \pmod 2$$

for some hidden bitstring $s \in \{0, 1\}^n$, where $s \cdot x = s_0 x_0 \oplus s_1 x_1 \oplus \dots \oplus s_{n-1} x_{n-1}$ is the bitwise inner product modulo 2.

Our job is to recover the secret string $s$.

- A classical algorithm needs $n$ oracle queries in the worst case.
- The Bernstein-Vazirani algorithm recovers $s$ with certainty using **a single** oracle query.

We will build the algorithm in three tasks:
1. Implement the oracle $U_f$ for a given secret $s$.
2. Implement the circuit that runs the algorithm (the same circuit as Deutsch-Jozsa).
3. Implement the algorithm that runs the circuit and returns the secret string $s$.

In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

print("Qiskit imported successfully!")

Qiskit imported successfully!


### Helper Functions & Guardrails

The oracle acts on $n$ input qubits **plus one ancilla qubit** used for phase kickback. We adopt the convention that the ancilla is the last qubit, at index $n$.

The helpers below are used by the test cells:

- `validate_secret(s)` checks that `s` is a non-empty binary string.
- `validate_oracle_inputs(n, oracle)` checks that `n` is valid and the oracle acts on `n + 1` qubits.
- `all_basis_states(n)` returns every $n$-bit input as a bitstring.
- `inner_product_mod2(s, x)` computes $s \cdot x \pmod 2$.
- `oracle_value(oracle, n, x)` evaluates the oracle classically by simulating $U_f |x\rangle|0\rangle$ and reading the ancilla.

In [2]:
def validate_secret(s: str) -> None:
    """Guardrail: checks s is a non-empty binary string."""
    assert isinstance(s, str), "The secret must be a string."
    assert len(s) >= 1, "The secret must have at least one bit."
    assert set(s).issubset({'0', '1'}), (
        f"Secret '{s}' must only contain '0' and '1'."
    )


def validate_oracle_inputs(n: int, oracle: QuantumCircuit | None = None) -> None:
    """Guardrail: checks n is a positive integer and the oracle has n + 1 qubits."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    if oracle is not None:
        assert oracle.num_qubits == n + 1, (
            f"Oracle must act on n + 1 = {n + 1} qubits (n inputs plus one ancilla), "
            f"but it acts on {oracle.num_qubits}."
        )


def all_basis_states(n: int) -> list[str]:
    """Returns every n-bit computational basis state as a bitstring."""
    return [format(i, f'0{n}b') for i in range(2 ** n)]


def inner_product_mod2(s: str, x: str) -> int:
    """Returns the bitwise inner product s . x modulo 2."""
    assert len(s) == len(x), "s and x must have the same length."
    return sum(int(a) * int(b) for a, b in zip(s, x)) % 2


def oracle_value(oracle: QuantumCircuit, n: int, x: str) -> int:
    """
    Evaluates the oracle f on the input x.

    Simulates U_f |x>|0>, where the ancilla is qubit n, and returns f(x).
    """
    validate_oracle_inputs(n, oracle)
    assert len(x) == n, f"Input '{x}' must have length n = {n}."
    assert set(x).issubset({'0', '1'}), f"Input '{x}' must be a binary string."

    circuit = QuantumCircuit(n + 1)
    for i, bit in enumerate(reversed(x)):
        if bit == '1':
            circuit.x(i)
    circuit.compose(oracle, inplace=True)

    statevector = Statevector(circuit)
    return int(round(statevector.probabilities([n])[1]))


print("Helper functions and guardrails loaded successfully!")

Helper functions and guardrails loaded successfully!


## Task 1: Building the Oracle

An oracle is a reversible circuit $U_f$ that acts on $|x\rangle|y\rangle$ as

$$U_f |x\rangle |y\rangle = |x\rangle |y \oplus f(x)\rangle.$$

For the Bernstein-Vazirani problem the oracle encodes the function $f(x) = s \cdot x \pmod 2$, which is linear in the input bits. Such a function is implemented by a set of CNOT gates:

- For **every** qubit $i$ where $s_i = 1$, apply a CNOT from input qubit $i$ to the ancilla.
- Qubits where $s_i = 0$ get no gate.

The ancilla will be prepared in $|-\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$, so that flipping it becomes a phase:

$$U_f |x\rangle |-\rangle = (-1)^{f(x)} |x\rangle |-\rangle.$$

### Task 1.1: Implement the Oracle

Implement `bv_oracle(s: str) -> QuantumCircuit`.

- `s`: the secret bitstring of length $n$.
- Returns an `(n + 1)`-qubit circuit implementing $f(x) = s \cdot x \pmod 2$, with the ancilla at index $n$.

> Hint: in a bitstring `s_{n-1} ... s_1 s_0`, the character `s[i]` corresponds to qubit `i`, so `s[-1]` is qubit 0 (Qiskit uses little-endian ordering). A CNOT with control `i` and target `n` adds bit `i` of the input to the ancilla.

In [15]:
def bv_oracle(s: str) -> QuantumCircuit:
    """
    Builds the oracle f(x) = s . x mod 2 on n input qubits plus one ancilla.

    Args:
        s: The secret bitstring of length n.

    Returns:
        QuantumCircuit: An (n + 1)-qubit circuit implementing the oracle,
        with the ancilla at index n.
    """
    validate_secret(s)
    n = len(s)
    qc = QuantumCircuit(n+1)
    for i in range(n):
        if s[n-i-1]=='1':
            qc.cx(i, n)

    # TODO: build the (n + 1)-qubit oracle circuit
    return qc

### Task 1.2: Test the Oracle

The cell below checks that your oracle implements $f(x) = s \cdot x \pmod 2$ for every input $x$ and several secrets $s$.

In [16]:
for s in ["0", "1", "10", "101", "0110", "1111", "1001"]:
    n = len(s)
    oracle = bv_oracle(s)
    for x in all_basis_states(n):
        expected = inner_product_mod2(s, x)
        observed = oracle_value(oracle, n, x)
        assert observed == expected, (
            f"Oracle for s={s} returned {observed} for x={x}, expected {expected}"
        )
print("Oracle passed all tests!")

Oracle passed all tests!


## Task 2: Building the Algorithm Circuit

Implement `bv_circuit(n: int, oracle: QuantumCircuit) -> QuantumCircuit`.

- `n`: number of input qubits.
- `oracle`: an `(n + 1)`-qubit oracle circuit, as built above.
- Returns a circuit implementing the Bernstein-Vazirani algorithm, with `n` classical bits measuring the input register.

This is **the same circuit as Deutsch-Jozsa**. The ancilla (the last qubit) provides phase kickback, and only the `n` input qubits are measured.

```
                      ┌───────┐
q_0:     |0> ─ ─ ─ H ─┤       ├─ H ─ M
q_1:     |0> ─ ─ ─ H ─┤       ├─ H ─ M
  ⋮             ⋮      │  U_f  │  ⋮   ⋮
q_(n-1): |0> ─ ─ ─ H ─┤       ├─ H ─ M
q_n:     |0> ─ X ─ H ─┤       ├─ H
                      └───────┘
```

> Why it works: after the first layer of Hadamards and the phase kickback, the input register is in $\frac{1}{\sqrt{2^n}} \sum_x (-1)^{s \cdot x} |x\rangle$. Applying $H^{\otimes n}$ again collapses this to $|s\rangle$.

In [17]:
def bv_circuit(n: int, oracle: QuantumCircuit) -> QuantumCircuit:
    """
    Builds a circuit implementing the Bernstein-Vazirani algorithm.

    Args:
        n: Number of input qubits.
        oracle: An (n + 1)-qubit oracle circuit.

    Returns:
        QuantumCircuit: The Bernstein-Vazirani circuit with n classical bits
        measuring the input register.
    """
    validate_oracle_inputs(n, oracle)
    qc = QuantumCircuit(n+1, n)
    qc.x(n)
    for i in range(n+1):
        qc.h(i)
    qc = qc.compose(oracle)
    for i in range(n+1):
        qc.h(i)
    qc.measure(range(n), range(n))

    # TODO: implement the circuit shown in the diagram above
    return qc

## Task 3: Running the Algorithm

Implement `bernstein_vazirani(n: int, oracle: QuantumCircuit) -> str`.

- `n`: number of input qubits.
- `oracle`: an `(n + 1)`-qubit oracle circuit.
- Returns the secret bitstring `s` of length `n`.

Because the output $|s\rangle$ is deterministic, every shot of the simulation yields the same bitstring. Build the circuit with `bv_circuit`, run it on `AerSimulator`, and return the measured bitstring.

> Hint: `get_counts()` returns a dictionary mapping measured bitstrings to their frequencies. The most frequent key is the secret.

In [18]:
def bernstein_vazirani(n: int, oracle: QuantumCircuit) -> str:
    """
    Runs the Bernstein-Vazirani algorithm and returns the secret string s.

    Args:
        n: Number of input qubits.
        oracle: An (n + 1)-qubit oracle circuit.

    Returns:
        str: The measured secret bitstring of length n.
    """
    validate_oracle_inputs(n, oracle)

    validate_oracle_inputs(n, oracle)
    qc = bv_circuit(n, oracle)
    simulator = AerSimulator()
    job = simulator.run(qc)
    result = job.result()
    counts = result.get_counts()


    # TODO: build the circuit with bv_circuit, run it on AerSimulator,
    # and return the most frequent measured bitstring.
    return max(counts, key=counts.get)

### Final Test

The cell below recovers a variety of secrets using your implementation. It must pass for the lab to be complete.

In [19]:
for s in ["0", "1", "10", "11", "101", "110", "0110", "1001", "1101", "10110", "111000"]:
    n = len(s)
    oracle = bv_oracle(s)
    recovered = bernstein_vazirani(n, oracle)
    assert recovered == s, (
        f"Failed to recover secret s={s}: got {recovered}"
    )

print("Bernstein-Vazirani passed all tests!")

Bernstein-Vazirani passed all tests!
